<a href="https://colab.research.google.com/github/Syed487/Veritas-AI-Hackathon/blob/main/Veritas_AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import bz2
import os

extracted_files = []

for filename in uploaded.keys():
    output_name = filename.removesuffix(".bz2")

    with bz2.open(filename, "rt", encoding="utf-8") as file:
        with open(output_name, "w", encoding="utf-8") as output:
            output.write(file.read())

    extracted_files.append(output_name)
    print("Extracted:", output_name)

In [ ]:
for filename in extracted_files:
    with open(filename, "r", encoding="utf-8") as file:
        print("\nFILE:", filename)
        for i in range(3):
            print(file.readline().strip())

In [ ]:
import pandas as pd

# Read all five files
dataframes = []

for filename in extracted_files:
    df = pd.read_csv(
        filename,
        sep="\t",
        header=None,
        names=["sentence_id", "language", "sentence"]
    )
    dataframes.append(df)

# Combine into one dataset
data = pd.concat(dataframes, ignore_index=True)

# Check how many sentences we have per language
print(data.groupby("language").size())

In [ ]:
# Select 7,000 sentences from each language
data = data.groupby("language", group_keys=False).sample(
    n=7000,
    random_state=42
)

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print("Total sentences:", len(data))
print("\nSentences per language:")
print(data["language"].value_counts())

# Show sample sentences
display(data.head(10))

In [ ]:
from sklearn.model_selection import train_test_split

# X contains the sentences (the input)
X = data["sentence"]

# y contains the correct language labels (the answers)
y = data["language"]

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training sentences:", len(X_train))
print("Testing sentences:", len(X_test))

print("\nTraining language distribution:")
print(y_train.value_counts())

print("\nTesting language distribution:")
print(y_test.value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create our text-to-number converter
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 3),
    max_features=50000
)

# Learn patterns from training sentences
X_train_tfidf = vectorizer.fit_transform(X_train)

# Convert testing sentences using the same patterns
X_test_tfidf = vectorizer.transform(X_test)

# Display the results
print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)
print("Number of character patterns:", len(vectorizer.get_feature_names_out()))

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the AI model
model = LogisticRegression(max_iter=1000)

# Train the model using our training data
model.fit(X_train_tfidf, y_train)

print("Model training completed!")

In [ ]:
from sklearn.metrics import accuracy_score

# Ask the model to predict the languages
y_pred = model.predict(X_test_tfidf)

# Compare predictions with correct answers
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

In [ ]:
from sklearn.metrics import classification_report

# Names for our language codes
language_names = {
    "eng": "English",
    "zsm": "Malay",
    "ind": "Indonesian",
    "spa": "Spanish",
    "fra": "French"
}

# Generate the classification report
print(classification_report(
    y_test,
    y_pred,
    labels=["eng", "zsm", "ind", "spa", "fra"],
    target_names=["English", "Malay", "Indonesian", "Spanish", "French"]
))

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Keep the languages in a fixed order
labels = ["eng", "zsm", "ind", "spa", "fra"]

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=labels)

# Display it as a readable table
confusion_table = pd.DataFrame(
    cm,
    index=["Actual " + language_names[x] for x in labels],
    columns=["Predicted " + language_names[x] for x in labels]
)

display(confusion_table)

In [ ]:
# Create a table of actual and predicted languages
results = pd.DataFrame({
    "Sentence": X_test.values,
    "Actual language": y_test.values,
    "Predicted language": y_pred
})

# Keep only incorrect predictions
mistakes = results[
    results["Actual language"] != results["Predicted language"]
]

# Show 15 examples
display(mistakes.head(15))

print("Total incorrect predictions:", len(mistakes))

In [ ]:
from sklearn.model_selection import train_test_split

# Split the original training data again
X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("New training data:", len(X_train_new))
print("Validation data:", len(X_val))
print("Final test data:", len(X_test))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create an improved TF-IDF converter
vectorizer_v2 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 5),
    max_features=50000
)

# Learn patterns using training sentences only
X_train_v2 = vectorizer_v2.fit_transform(X_train_new)

# Convert validation sentences
X_val_v2 = vectorizer_v2.transform(X_val)

print("Training data shape:", X_train_v2.shape)
print("Validation data shape:", X_val_v2.shape)
print("Number of patterns:", len(vectorizer_v2.get_feature_names_out()))

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the second model
model_v2 = LogisticRegression(max_iter=1000)

# Train it using the new training data
model_v2.fit(X_train_v2, y_train_new)

print("Improved model training completed!")

In [ ]:
from sklearn.metrics import accuracy_score

# Predict languages using Model 2
y_val_pred = model_v2.predict(X_val_v2)

# Calculate validation accuracy
accuracy_v2 = accuracy_score(y_val, y_val_pred)

print("Improved Model Accuracy:", round(accuracy_v2 * 100, 2), "%")

In [ ]:
# Convert validation sentences using the original TF-IDF converter
X_val_original = vectorizer.transform(X_val)

# Predict using the original model
original_val_pred = model.predict(X_val_original)

# Calculate both validation accuracies
original_val_accuracy = accuracy_score(y_val, original_val_pred)
improved_val_accuracy = accuracy_score(y_val, y_val_pred)

print("Original Model:", round(original_val_accuracy * 100, 2), "%")
print("Improved Model:", round(improved_val_accuracy * 100, 2), "%")

print(
    "Difference:",
    round((improved_val_accuracy - original_val_accuracy) * 100, 2),
    "percentage points"
)

In [ ]:
from sklearn.metrics import classification_report

labels = ["eng", "zsm", "ind", "spa", "fra"]

print("ORIGINAL MODEL")
print(classification_report(
    y_val,
    original_val_pred,
    labels=labels,
    target_names=["English", "Malay", "Indonesian", "Spanish", "French"]
))

print("IMPROVED MODEL")
print(classification_report(
    y_val,
    y_val_pred,
    labels=labels,
    target_names=["English", "Malay", "Indonesian", "Spanish", "French"]
))



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create the original 1–3 character model
vectorizer_v1_fair = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 3),
    max_features=50000
)

# Learn patterns using only the training data
X_train_v1_fair = vectorizer_v1_fair.fit_transform(X_train_new)

# Convert validation sentences using the same patterns
X_val_v1_fair = vectorizer_v1_fair.transform(X_val)

# Train the model
model_v1_fair = LogisticRegression(max_iter=1000)
model_v1_fair.fit(X_train_v1_fair, y_train_new)

# Make predictions on validation data
y_val_pred_v1_fair = model_v1_fair.predict(X_val_v1_fair)

# Compare both models
print("ORIGINAL MODEL (1–3 characters)")
print("Accuracy:", accuracy_score(y_val, y_val_pred_v1_fair))

print("\nIMPROVED MODEL (1–5 characters)")
print("Accuracy:", accuracy_score(y_val, y_val_pred))

print("\nORIGINAL MODEL REPORT")
print(classification_report(y_val, y_val_pred_v1_fair))

print("\nIMPROVED MODEL REPORT")
print(classification_report(y_val, y_val_pred))

In [ ]:
# Create the final TF-IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

final_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 5),
    max_features=50000
)

# Convert all training sentences into numerical features
X_train_final = final_vectorizer.fit_transform(X_train)

# Create and train the final model
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_final, y_train)

print("Final model training completed!")


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Convert the test sentences into numerical features
X_test_final = final_vectorizer.transform(X_test)

# Predict the languages
final_predictions = final_model.predict(X_test_final)

# Calculate final accuracy
final_accuracy = accuracy_score(y_test, final_predictions)

print("FINAL TEST ACCURACY:", final_accuracy)

print("\nFINAL CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    final_predictions,
    labels=["eng", "zsm", "ind", "spa", "fra"],
    target_names=["English", "Malay", "Indonesian", "Spanish", "French"]
))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Define the language order
labels = ["eng", "zsm", "ind", "spa", "fra"]
names = ["English", "Malay", "Indonesian", "Spanish", "French"]

# Calculate the confusion matrix
cm = confusion_matrix(y_test, final_predictions, labels=labels)

# Display it as a graph
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=names
)

display.plot(xticks_rotation=30)
plt.title("Final Language Identification Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Language names and their support queues
support_queues = {
    "eng": "English Support",
    "zsm": "Malay Support",
    "ind": "Indonesian Support",
    "spa": "Spanish Support",
    "fra": "French Support"
}

# Create the LanguageLink function
def language_link(sentence):

    sentence_features = final_vectorizer.transform([sentence])

    prediction = final_model.predict(sentence_features)[0]

    probabilities = final_model.predict_proba(sentence_features)[0]
    confidence = max(probabilities)

    # Improved confidence threshold
    if confidence < 0.75:
        queue = "General Support"
    else:
        queue = support_queues[prediction]

    print("VERITAS LANGUAGELINK")
    print("--------------------")
    print("Input:", sentence)
    print("Detected language:", prediction)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("Assigned queue:", queue)

In [ ]:
language_link("Saya suka makan nasi goreng.")

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr

language_names = {
    "eng": "English",
    "zsm": "Malay",
    "ind": "Indonesian",
    "spa": "Spanish",
    "fra": "French"
}

def detect_and_route(sentence):

    if not sentence or not sentence.strip():
        return "Please enter a sentence."

    features = final_vectorizer.transform([sentence])

    prediction = final_model.predict(features)[0]
    probabilities = final_model.predict_proba(features)[0]
    confidence = max(probabilities)

    if confidence < 0.75:
        queue = "General Support"
        status = "Manual review recommended"
    else:
        queue = support_queues[prediction]
        status = "Language queue assigned"

    return f"""
## Detection Results

**Detected Language:** {language_names[prediction]}

**Confidence:** {confidence * 100:.2f}%

**Assigned Queue:** {queue}

**Status:** {status}

---
*Confidence is a model estimate, not a guarantee of correctness.*
"""

with gr.Blocks(title="Veritas LanguageLink") as app:

    gr.Markdown("# Veritas LanguageLink")
    gr.Markdown(
        "AI-powered language identification and smart support routing."
    )

    sentence_input = gr.Textbox(
        label="Enter your sentence",
        placeholder="Type a sentence...",
        lines=3
    )

    with gr.Row():
        detect_button = gr.Button("Detect Language")
        clear_button = gr.ClearButton()

    result_output = gr.Markdown()

    detect_button.click(
        detect_and_route,
        inputs=sentence_input,
        outputs=result_output
    )

    clear_button.add(sentence_input)
    clear_button.add(result_output)

    gr.Markdown(
        "Supported languages: English, Malay, Indonesian, Spanish and French."
    )

app.launch(share=True)

In [58]:
import re
from collections import Counter

def detect_mixed_language(sentence):

    words = re.findall(r"\b[\w']+\b", sentence)

    if len(words) < 3:
        return {
            "result": "Message too short",
            "languages": [],
            "chunks": []
        }

    chunks = []

    for i in range(len(words) - 2):
        chunks.append(" ".join(words[i:i+3]))

    results = []

    for chunk in chunks:

        features = final_vectorizer.transform([chunk])
        prediction = final_model.predict(features)[0]

        probabilities = final_model.predict_proba(features)[0]
        confidence = max(probabilities)

        results.append({
            "text": chunk,
            "language": prediction,
            "confidence": confidence
        })

    reliable_results = [
        r for r in results if r["confidence"] >= 0.55
    ]

    language_counts = Counter(
        r["language"] for r in reliable_results
    )

    detected_languages = set(language_counts.keys())

    # Handle Malay and Indonesian similarity
    if detected_languages.issubset({"zsm", "ind"}):

        if language_counts:
            dominant = language_counts.most_common(1)[0][0]
            detected_languages = {dominant}

    if len(detected_languages) >= 2:
        status = "Possible mixed-language message detected"
    elif len(detected_languages) == 1:
        status = "One dominant language detected"
    else:
        status = "Unable to identify language reliably"

    languages = [
        language_names[lang]
        for lang in sorted(detected_languages)
    ]

    return {
        "result": status,
        "languages": languages,
        "chunks": results
    }

print("Detector updated with Malay-Indonesian safeguard!")

Detector updated with Malay-Indonesian safeguard!


In [ ]:
test_message = (
    "I really suka makan nasi lemak because "
    "it tastes very good and I love it."
)

result = detect_mixed_language(test_message)

print("RESULT:", result["result"])
print("LANGUAGES:", result.get("languages", []))

print("\nCHUNK ANALYSIS:")

for item in result.get("chunks", []):
    print(
        item["text"],
        "→",
        language_names[item["language"]],
        "|",
        round(item["confidence"] * 100, 2),
        "%"
    )

In [ ]:
test_cases = {
    "English only": "I really enjoy learning new things every day.",
    "Malay only": "Saya suka makan nasi lemak bersama keluarga.",
    "Indonesian only": "Saya suka belajar bahasa Indonesia setiap hari.",
    "Spanish only": "Me gusta aprender cosas nuevas todos los días.",
    "French only": "J'aime apprendre de nouvelles choses chaque jour.",
    "English and Malay": "I really suka makan nasi lemak because it tastes good."
}

for name, sentence in test_cases.items():

    result = detect_mixed_language(sentence)

    print("\nTEST:", name)
    print("Expected:", name)
    print("Detected:", result["result"])
    print("Languages:", result.get("languages", []))

In [ ]:
mixed_test_cases = {
    "English + Malay": (
        "I want to pergi makan with my friends today."
    ),
    "English + Indonesian": (
        "I want to makan nasi goreng today."
    ),
    "English + Spanish": (
        "I love aprender nuevos idiomas every day."
    ),
    "English + French": (
        "I enjoy apprendre le français."
    ),
    "Malay only": (
        "Saya pergi ke sekolah bersama kawan saya."
    ),
    "Indonesian only": (
        "Saya pergi ke sekolah bersama teman saya."
    )
}

for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result.get("languages", []))

In [51]:
mixed_test_cases = {
    "English + Malay": "I want to pergi makan with my friends today.",
    "English + Indonesian": "I want to makan nasi goreng today.",
    "English + Spanish": "I love aprender nuevos idiomas every day.",
    "English + French": "I enjoy apprendre le français.",
    "Malay only": "Saya pergi ke sekolah bersama kawan saya.",
    "Indonesian only": "Saya pergi ke sekolah bersama teman saya."
}

for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result.get("languages", []))


EXPECTED: English + Malay
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Indonesian
DETECTED: One dominant language detected
LANGUAGES: ['English']

EXPECTED: English + Spanish
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Spanish']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: Possible mixed-language message detected
LANGUAGES: ['Indonesian', 'Malay']

EXPECTED: Indonesian only
DETECTED: Possible mixed-language message detected
LANGUAGES: ['Indonesian', 'Malay']


In [52]:
result = detect_mixed_language(
    "Saya pergi ke sekolah bersama kawan saya."
)

for chunk in result["chunks"]:
    print(
        chunk["text"],
        "→",
        chunk["language"],
        f'{chunk["confidence"]:.2%}'
    )

Saya pergi ke → zsm 71.95%
pergi ke sekolah → ind 60.35%
ke sekolah bersama → zsm 58.97%
sekolah bersama kawan → zsm 86.94%
bersama kawan saya → zsm 93.30%


In [56]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result["languages"])


EXPECTED: English + Malay
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Indonesian
DETECTED: Unable to identify language reliably
LANGUAGES: []

EXPECTED: English + Spanish
DETECTED: One dominant language detected
LANGUAGES: ['Spanish']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']

EXPECTED: Indonesian only
DETECTED: Possible mixed-language message detected
LANGUAGES: ['Indonesian', 'Malay']


In [57]:
result = detect_mixed_language(
    "Saya pergi ke sekolah bersama teman saya."
)

for chunk in result["chunks"]:
    print(
        chunk["text"],
        "→",
        language_names[chunk["language"]],
        f'{chunk["confidence"]:.2%}'
    )

Saya pergi ke → Malay 71.95%
pergi ke sekolah → Indonesian 60.35%
ke sekolah bersama → Malay 58.97%
sekolah bersama teman → Indonesian 59.76%
bersama teman saya → Indonesian 54.85%


In [59]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result["languages"])


EXPECTED: English + Malay
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Indonesian
DETECTED: One dominant language detected
LANGUAGES: ['English']

EXPECTED: English + Spanish
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Spanish']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']

EXPECTED: Indonesian only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']


In [60]:
test_sentence = "I want to pergi makan with my friends today."

features = final_vectorizer.transform([test_sentence])

probabilities = final_model.predict_proba(features)[0]

for language, probability in zip(
    final_model.classes_, probabilities
):
    print(
        language_names[language],
        f"{probability:.2%}"
    )

English 91.29%
French 0.60%
Indonesian 3.73%
Spanish 0.56%
Malay 3.82%


In [61]:
import re

test_sentence = "I want to pergi makan with my friends today."

words = re.findall(r"\b[\w']+\b", test_sentence)

for word in words:

    features = final_vectorizer.transform([word])

    probabilities = final_model.predict_proba(features)[0]

    best_index = probabilities.argmax()

    language = final_model.classes_[best_index]

    confidence = probabilities[best_index]

    print(
        f"{word:12} → "
        f"{language_names[language]:12} "
        f"{confidence:.2%}"
    )

I            → Indonesian   57.66%
want         → English      42.60%
to           → English      66.00%
pergi        → Indonesian   72.35%
makan        → Indonesian   79.24%
with         → English      95.83%
my           → English      62.02%
friends      → French       35.75%
today        → English      36.29%


In [62]:
for i in range(len(words) - 1):

    chunk = " ".join(words[i:i+2])

    features = final_vectorizer.transform([chunk])

    probabilities = final_model.predict_proba(features)[0]

    best_index = probabilities.argmax()

    language = final_model.classes_[best_index]

    confidence = probabilities[best_index]

    print(
        f"{chunk:20} → "
        f"{language_names[language]:12} "
        f"{confidence:.2%}"
    )

I want               → English      80.94%
want to              → English      78.13%
to pergi             → Indonesian   40.88%
pergi makan          → Indonesian   76.39%
makan with           → Indonesian   41.25%
with my              → English      87.40%
my friends           → English      61.94%
friends today        → English      56.69%


In [63]:
for size in [2, 3, 4]:

    print(f"\n========== {size}-WORD GROUPS ==========")

    for i in range(len(words) - size + 1):

        chunk = " ".join(words[i:i+size])

        features = final_vectorizer.transform([chunk])

        probabilities = final_model.predict_proba(features)[0]

        best_index = probabilities.argmax()

        language = final_model.classes_[best_index]

        confidence = probabilities[best_index]

        print(
            f"{chunk:35} → "
            f"{language_names[language]:12} "
            f"{confidence:.2%}"
        )


========== 2-WORD GROUPS ==========
I want                              → English      80.94%
want to                             → English      78.13%
to pergi                            → Indonesian   40.88%
pergi makan                         → Indonesian   76.39%
makan with                          → Indonesian   41.25%
with my                             → English      87.40%
my friends                          → English      61.94%
friends today                       → English      56.69%

========== 3-WORD GROUPS ==========
I want to                           → English      91.41%
want to pergi                       → English      58.80%
to pergi makan                      → Indonesian   70.64%
pergi makan with                    → Indonesian   57.32%
makan with my                       → English      47.49%
with my friends                     → English      88.65%
my friends today                    → English      75.27%

========== 4-WORD GROUPS ==========
I want to pergi    

In [69]:
import re
from collections import defaultdict

def detect_mixed_language_v2(sentence):

    words = re.findall(r"\b[\w']+\b", sentence)

    if len(words) < 2:
        return {
            "result": "Message too short",
            "languages": [],
            "details": []
        }

    evidence = defaultdict(float)
    details = []

    for size in [2, 3]:

        for i in range(len(words) - size + 1):

            chunk = " ".join(words[i:i+size])

            features = final_vectorizer.transform([chunk])

            probabilities = final_model.predict_proba(features)[0]

            best_index = probabilities.argmax()

            language = final_model.classes_[best_index]

            confidence = probabilities[best_index]

            details.append({
                "chunk": chunk,
                "language": language_names[language],
                "confidence": confidence
            })

            # Only count reasonably confident predictions
            if confidence >= 0.55:

                evidence[language] += confidence

    # Rank languages by total evidence
    ranked_languages = sorted(
        evidence.items(),
        key=lambda x: x[1],
        reverse=True
    )

    detected_languages = []

    if ranked_languages:

        strongest_score = ranked_languages[0][1]

        for language, score in ranked_languages:

            if score >= strongest_score * 0.80:
                detected_languages.append(language)

    if len(detected_languages) >= 2:
        status = "Possible mixed-language message detected"
    elif len(detected_languages) == 1:
        status = "One dominant language detected"
    else:
        status = "Unable to identify language reliably"

    return {
        "result": status,
        "languages": [
            language_names[lang] for lang in detected_languages
        ],
        "details": details
    }

print("Combined mixed-language detector ready!")

Combined mixed-language detector ready!


In [66]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language_v2(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result["languages"])


EXPECTED: English + Malay
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Indonesian
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Spanish
DETECTED: Possible mixed-language message detected
LANGUAGES: ['Spanish', 'English']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']

EXPECTED: Indonesian only
DETECTED: Possible mixed-language message detected
LANGUAGES: ['Indonesian', 'Malay']


In [67]:
sentence = "Saya pergi ke sekolah bersama teman saya."

result = detect_mixed_language_v2(sentence)

language_scores = {}

for item in result["details"]:

    language = item["language"]
    confidence = item["confidence"]

    if confidence >= 0.55:
        language_scores[language] = (
            language_scores.get(language, 0) + confidence
        )

print("LANGUAGE EVIDENCE:")

for language, score in sorted(
    language_scores.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{language}: {score:.3f}")

LANGUAGE EVIDENCE:
Indonesian: 2.686
Malay: 2.065


In [70]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language_v2(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result["languages"])



EXPECTED: English + Malay
DETECTED: One dominant language detected
LANGUAGES: ['English']

EXPECTED: English + Indonesian
DETECTED: One dominant language detected
LANGUAGES: ['English']

EXPECTED: English + Spanish
DETECTED: One dominant language detected
LANGUAGES: ['Spanish']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']

EXPECTED: Indonesian only
DETECTED: One dominant language detected
LANGUAGES: ['Indonesian']


In [73]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language_v2(sentence)

    scores = {}

    for item in result["details"]:

        language = item["language"]
        confidence = item["confidence"]

        if confidence >= 0.55:
            scores[language] = (
                scores.get(language, 0) + confidence
            )

    print("\nEXPECTED:", name)

    for language, score in sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(language, round(score, 3))


EXPECTED: English + Malay
English 6.792
Indonesian 2.044

EXPECTED: English + Indonesian
English 2.505
Indonesian 1.258

EXPECTED: English + Spanish
Spanish 3.432
English 2.109

EXPECTED: English + French
French 3.078

EXPECTED: Malay only
Malay 5.642
Indonesian 1.341

EXPECTED: Indonesian only
Indonesian 2.686
Malay 2.065


In [74]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language_v2(sentence)

    print("\n==========", name, "==========")

    print("FULL SENTENCE:")

    features = final_vectorizer.transform([sentence])
    prediction = final_model.predict(features)[0]

    print(language_names[prediction])

    print("\nCHUNK PREDICTIONS:")

    for item in result["details"]:
        print(
            item["chunk"],
            "→",
            item["language"],
            f'{item["confidence"]:.1%}'
        )


========== English + Malay ==========
FULL SENTENCE:
English

CHUNK PREDICTIONS:
I want → English 80.9%
want to → English 78.1%
to pergi → Indonesian 40.9%
pergi makan → Indonesian 76.4%
makan with → Indonesian 41.2%
with my → English 87.4%
my friends → English 61.9%
friends today → English 56.7%
I want to → English 91.4%
want to pergi → English 58.8%
to pergi makan → Indonesian 70.6%
pergi makan with → Indonesian 57.3%
makan with my → English 47.5%
with my friends → English 88.6%
my friends today → English 75.3%

========== English + Indonesian ==========
FULL SENTENCE:
English

CHUNK PREDICTIONS:
I want → English 80.9%
want to → English 78.1%
to makan → Indonesian 68.3%
makan nasi → Indonesian 57.5%
nasi goreng → Indonesian 32.2%
goreng today → English 53.5%
I want to → English 91.4%
want to makan → English 46.0%
to makan nasi → Indonesian 53.1%
makan nasi goreng → Indonesian 55.0%
nasi goreng today → English 43.1%

========== English + Spanish ==========
FULL SENTENCE:
English

CHU

In [75]:
def detect_mixed_language_v3(sentence):

    words = re.findall(r"\b[\w']+\b", sentence)

    if len(words) < 3:
        return {
            "result": "Message too short",
            "languages": [],
            "details": []
        }

    details = []

    # Analyse 2-word and 3-word chunks
    for size in [2, 3]:

        for i in range(len(words) - size + 1):

            chunk = " ".join(words[i:i+size])

            features = final_vectorizer.transform([chunk])
            probabilities = final_model.predict_proba(features)[0]

            index = probabilities.argmax()

            language = final_model.classes_[index]
            confidence = probabilities[index]

            details.append({
                "chunk": chunk,
                "language": language,
                "confidence": confidence
            })

    # Keep confident predictions
    reliable = [
        item for item in details
        if item["confidence"] >= 0.60
    ]

    # Count language predictions
    counts = Counter(
        item["language"] for item in reliable
    )

    # Find languages with at least two predictions
    candidates = {
        lang for lang, count in counts.items()
        if count >= 2
    }

    # Do not interpret Malay/Indonesian confusion
    # as mixed-language evidence by itself
    if candidates.issubset({"zsm", "ind"}):
        if counts:
            dominant = counts.most_common(1)[0][0]
            candidates = {dominant}

    languages = [
        language_names[lang]
        for lang in sorted(candidates)
    ]

    if len(languages) >= 2:
        status = "Possible mixed-language message detected"
    elif len(languages) == 1:
        status = "One dominant language detected"
    else:
        status = "Unable to identify reliably"

    return {
        "result": status,
        "languages": languages,
        "details": details
    }


print("Transition-based detector ready!")

Transition-based detector ready!


In [76]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language_v3(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result["languages"])


EXPECTED: English + Malay
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Indonesian
DETECTED: One dominant language detected
LANGUAGES: ['English']

EXPECTED: English + Spanish
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Spanish']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']

EXPECTED: Indonesian only
DETECTED: One dominant language detected
LANGUAGES: ['Indonesian']


In [77]:
sentence = "I want to makan nasi goreng today."

chunks = [
    "I want",
    "want to",
    "to makan",
    "makan nasi",
    "nasi goreng",
    "goreng today"
]

for chunk in chunks:

    features = final_vectorizer.transform([chunk])

    probabilities = final_model.predict_proba(features)[0]

    print("\n", chunk)

    for language, probability in zip(
        final_model.classes_, probabilities
    ):
        print(
            language_names[language],
            f"{probability:.2%}"
        )


 I want
English 80.94%
French 2.34%
Indonesian 6.82%
Spanish 1.52%
Malay 8.38%

 want to
English 78.13%
French 10.20%
Indonesian 5.13%
Spanish 1.79%
Malay 4.74%

 to makan
English 6.75%
French 1.16%
Indonesian 68.33%
Spanish 8.28%
Malay 15.48%

 makan nasi
English 1.78%
French 1.60%
Indonesian 57.45%
Spanish 3.91%
Malay 35.25%

 nasi goreng
English 21.30%
French 8.39%
Indonesian 32.21%
Spanish 24.87%
Malay 13.23%

 goreng today
English 53.49%
French 3.13%
Indonesian 19.61%
Spanish 16.81%
Malay 6.96%


In [86]:
def detect_mixed_language_v4(sentence):

    words = re.findall(r"\b[\w']+\b", sentence)

    if len(words) < 3:
        return {
            "result": "Message too short",
            "languages": [],
            "warning": "Enter a longer message.",
            "details": []
        }

    details = []

    for size in [2, 3]:

        for i in range(len(words) - size + 1):

            chunk = " ".join(words[i:i+size])

            features = final_vectorizer.transform([chunk])

            probabilities = final_model.predict_proba(features)[0]

            best_index = probabilities.argmax()

            details.append({
                "chunk": chunk,
                "language": final_model.classes_[best_index],
                "confidence": probabilities[best_index],
                "probabilities": probabilities
            })

    language_counts = Counter()

    for item in details:

        if item["confidence"] >= 0.55:
            language_counts[item["language"]] += 1

    candidates = {
        lang for lang, count in language_counts.items()
        if count >= 2
    }

    detected_languages = set(candidates)

    if detected_languages.issubset({"zsm", "ind"}):
        if language_counts:
            dominant = language_counts.most_common(1)[0][0]
            detected_languages = {dominant}

    if len(detected_languages) >= 2:

        status = "Possible mixed-language message detected"
        warning = "Language identification may be less reliable for similar languages."

    elif len(detected_languages) == 1:

        status = "One dominant language detected"
        warning = "A second language may be present but could not be confirmed."

    else:

        status = "Unable to identify reliably"
        warning = "Try entering a longer message."

    return {
        "result": status,
        "languages": [
            language_names[lang]
            for lang in sorted(detected_languages)
        ],
        "warning": warning,
        "details": details
    }

In [79]:
for name, sentence in mixed_test_cases.items():

    result = detect_mixed_language_v4(sentence)

    print("\nEXPECTED:", name)
    print("DETECTED:", result["result"])
    print("LANGUAGES:", result["languages"])


EXPECTED: English + Malay
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Indonesian
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Indonesian']

EXPECTED: English + Spanish
DETECTED: Possible mixed-language message detected
LANGUAGES: ['English', 'Spanish']

EXPECTED: English + French
DETECTED: One dominant language detected
LANGUAGES: ['French']

EXPECTED: Malay only
DETECTED: One dominant language detected
LANGUAGES: ['Malay']

EXPECTED: Indonesian only
DETECTED: One dominant language detected
LANGUAGES: ['Indonesian']


In [80]:
sentence = "I enjoy apprendre le français."

words = re.findall(r"\b[\w']+\b", sentence)

for size in [2, 3]:

    print(f"\n===== {size}-WORD GROUPS =====")

    for i in range(len(words) - size + 1):

        chunk = " ".join(words[i:i+size])

        features = final_vectorizer.transform([chunk])

        probabilities = final_model.predict_proba(features)[0]

        print("\n", chunk)

        for language, probability in zip(
            final_model.classes_, probabilities
        ):

            if probability >= 0.10:
                print(
                    language_names[language],
                    f"{probability:.2%}"
                )


===== 2-WORD GROUPS =====

 I enjoy
English 14.82%
French 20.90%
Indonesian 15.25%
Spanish 39.84%

 enjoy apprendre
English 11.62%
French 44.04%
Spanish 32.93%

 apprendre le
French 84.37%

 le français
French 68.22%
Spanish 10.10%

===== 3-WORD GROUPS =====

 I enjoy apprendre
English 11.12%
French 43.99%
Spanish 33.77%

 enjoy apprendre le
French 65.29%
Spanish 21.77%

 apprendre le français
French 89.88%


In [81]:
test_sentences = [
    "I enjoy apprendre le français.",
    "I really enjoy learning new things because apprendre le français is interesting to me.",
    "I think learning languages is useful because apprendre le français is interesting."
]

for sentence in test_sentences:

    features = final_vectorizer.transform([sentence])

    probabilities = final_model.predict_proba(features)[0]

    print("\nSENTENCE:", sentence)

    for language, probability in sorted(
        zip(final_model.classes_, probabilities),
        key=lambda x: x[1],
        reverse=True
    ):

        if probability >= 0.05:
            print(language_names[language], f"{probability:.2%}")


SENTENCE: I enjoy apprendre le français.
French 83.21%
Spanish 9.42%

SENTENCE: I really enjoy learning new things because apprendre le français is interesting to me.
English 93.13%

SENTENCE: I think learning languages is useful because apprendre le français is interesting.
English 81.36%
French 13.00%


In [82]:
for sentence in test_sentences:

    result = detect_mixed_language_v4(sentence)

    print("\nSENTENCE:", sentence)
    print("RESULT:", result["result"])
    print("LANGUAGES:", result["languages"])


SENTENCE: I enjoy apprendre le français.
RESULT: One dominant language detected
LANGUAGES: ['French']

SENTENCE: I really enjoy learning new things because apprendre le français is interesting to me.
RESULT: Possible mixed-language message detected
LANGUAGES: ['English', 'French']

SENTENCE: I think learning languages is useful because apprendre le français is interesting.
RESULT: Possible mixed-language message detected
LANGUAGES: ['English', 'French']


In [87]:
expanded_test_cases = {

    # English + Malay
    "English + Malay 1": "I want to pergi makan with my friends today.",
    "English + Malay 2": "Can you tolong me with my homework?",
    "English + Malay 3": "I sudah finish my work.",

    # English + Indonesian
    "English + Indonesian 1": "I want to makan nasi goreng today.",
    "English + Indonesian 2": "Can you bantu me with this question?",
    "English + Indonesian 3": "I sedang belajar English today.",

    # English + Spanish
    "English + Spanish 1": "I love aprender nuevos idiomas every day.",
    "English + Spanish 2": "This is muy importante for my studies.",
    "English + Spanish 3": "I want to comer something delicious.",

    # English + French
    "English + French 1": "I really enjoy learning because apprendre le français is interesting.",
    "English + French 2": "I think bonjour is a beautiful French greeting.",
    "English + French 3": "Learning languages is fun and j'aime beaucoup le français.",

    # Single-language messages
    "English only": "I enjoy playing football with my friends every weekend.",
    "Malay only": "Saya suka bermain bola sepak bersama kawan saya.",
    "Indonesian only": "Saya suka bermain sepak bola bersama teman saya."
}

for name, sentence in expanded_test_cases.items():

    result = detect_mixed_language_v4(sentence)

    print("\n" + "=" * 55)
    print("TEST:", name)
    print("INPUT:", sentence)
    print("RESULT:", result["result"])
    print("DETECTED:", result["languages"])
    print("WARNING:", result["warning"])


TEST: English + Malay 1
INPUT: I want to pergi makan with my friends today.
RESULT: Possible mixed-language message detected
DETECTED: ['English', 'Indonesian']

TEST: English + Malay 2
INPUT: Can you tolong me with my homework?
RESULT: One dominant language detected
DETECTED: ['English']

TEST: English + Malay 3
INPUT: I sudah finish my work.
RESULT: Possible mixed-language message detected
DETECTED: ['English', 'Indonesian']

TEST: English + Indonesian 1
INPUT: I want to makan nasi goreng today.
RESULT: Possible mixed-language message detected
DETECTED: ['English', 'Indonesian']

TEST: English + Indonesian 2
INPUT: Can you bantu me with this question?
RESULT: One dominant language detected
DETECTED: ['English']

TEST: English + Indonesian 3
INPUT: I sedang belajar English today.
RESULT: One dominant language detected
DETECTED: ['Malay']

TEST: English + Spanish 1
INPUT: I love aprender nuevos idiomas every day.
RESULT: Possible mixed-language message detected
DETECTED: ['English', '

In [88]:
sentence = "Can you tolong me with my homework?"

words = re.findall(r"\b[\w']+\b", sentence)

for size in [2, 3]:

    print(f"\n===== {size}-WORD GROUPS =====")

    for i in range(len(words) - size + 1):

        chunk = " ".join(words[i:i+size])

        features = final_vectorizer.transform([chunk])

        probabilities = final_model.predict_proba(features)[0]

        print("\n", chunk)

        for language, probability in sorted(
            zip(final_model.classes_, probabilities),
            key=lambda x: x[1],
            reverse=True
        )[:3]:

            print(language_names[language], f"{probability:.2%}")


===== 2-WORD GROUPS =====

 Can you
English 83.73%
Indonesian 7.36%
Spanish 4.76%

 you tolong
English 62.67%
Indonesian 22.73%
French 6.67%

 tolong me
Indonesian 62.26%
Malay 14.00%
English 12.33%

 me with
English 95.33%
French 1.30%
Spanish 1.24%

 with my
English 87.40%
Indonesian 5.63%
Malay 3.79%

 my homework
English 81.49%
Indonesian 7.51%
Spanish 6.95%

===== 3-WORD GROUPS =====

 Can you tolong
English 77.60%
Indonesian 13.95%
Spanish 3.23%

 you tolong me
English 53.43%
Indonesian 31.10%
Malay 7.19%

 tolong me with
English 77.30%
Indonesian 12.00%
Malay 5.33%

 me with my
English 93.38%
Indonesian 2.11%
Malay 1.74%

 with my homework
English 93.58%
Indonesian 2.84%
Malay 1.52%


In [89]:
words_to_test = [
    "tolong",
    "bantu",
    "sedang",
    "belajar",
    "pergi",
    "makan",
    "comer",
    "aprender",
    "bonjour",
    "j'aime"
]

for word in words_to_test:

    features = final_vectorizer.transform([word])

    probabilities = final_model.predict_proba(features)[0]

    best_index = probabilities.argmax()

    print(
        word,
        "->",
        language_names[final_model.classes_[best_index]],
        f"({probabilities[best_index]:.2%})"
    )

tolong -> Indonesian (57.51%)
bantu -> Indonesian (63.27%)
sedang -> Indonesian (49.02%)
belajar -> Indonesian (70.26%)
pergi -> Indonesian (72.35%)
makan -> Indonesian (79.24%)
comer -> Spanish (56.30%)
aprender -> Spanish (51.10%)
bonjour -> French (70.62%)
j'aime -> French (92.83%)


In [90]:
malay_words = [
    "tolong",
    "bantu",
    "sedang",
    "belajar",
    "pergi",
    "makan"
]

for word in malay_words:

    features = final_vectorizer.transform([word])

    probabilities = final_model.predict_proba(features)[0]

    print("\nWORD:", word)

    for language, probability in sorted(
        zip(final_model.classes_, probabilities),
        key=lambda x: x[1],
        reverse=True
    ):

        print(
            language_names[language],
            f"{probability:.2%}"
        )



WORD: tolong
Indonesian 57.51%
English 14.20%
Spanish 11.37%
French 8.92%
Malay 8.00%

WORD: bantu
Indonesian 63.27%
Malay 19.33%
French 7.89%
Spanish 6.86%
English 2.66%

WORD: sedang
Indonesian 49.02%
Malay 39.32%
English 5.74%
Spanish 3.78%
French 2.15%

WORD: belajar
Indonesian 70.26%
Malay 12.78%
Spanish 12.39%
French 2.68%
English 1.88%

WORD: pergi
Indonesian 72.35%
Malay 14.48%
Spanish 4.75%
English 4.31%
French 4.12%

WORD: makan
Indonesian 79.24%
Malay 20.26%
Spanish 0.20%
English 0.17%
French 0.13%


In [98]:
def check_malay_indonesian_ambiguity(sentence):

    features = final_vectorizer.transform([sentence])

    probabilities = final_model.predict_proba(features)[0]

    probability_map = dict(
        zip(final_model.classes_, probabilities)
    )

    malay_score = probability_map["zsm"]
    indonesian_score = probability_map["ind"]

    difference = abs(malay_score - indonesian_score)

    # Only check ambiguity when at least one
    # of the two languages has meaningful probability.
    if max(malay_score, indonesian_score) < 0.30:

        status = "Not relevant"

    elif difference <= 0.20:

        status = "Uncertain"

    else:

        status = "Clearer prediction"

    return {
        "status": status,
        "malay_probability": round(malay_score * 100, 2),
        "indonesian_probability": round(indonesian_score * 100, 2)
    }


print("Ambiguity checker corrected!")

Ambiguity checker corrected!


In [92]:
ambiguity_tests = {
    "Malay sentence": "Saya suka bermain bola sepak bersama kawan saya.",
    "Indonesian sentence": "Saya suka bermain sepak bola bersama teman saya.",
    "Malay example": "Saya sedang belajar bahasa Melayu.",
    "Indonesian example": "Saya sedang belajar bahasa Indonesia."
}

for name, sentence in ambiguity_tests.items():

    print("\nTEST:", name)
    print("INPUT:", sentence)
    print(
        "CHECK:",
        check_malay_indonesian_ambiguity(sentence)
    )


TEST: Malay sentence
INPUT: Saya suka bermain bola sepak bersama kawan saya.
CHECK: Malay/Indonesian distinction is clearer

TEST: Indonesian sentence
INPUT: Saya suka bermain sepak bola bersama teman saya.
CHECK: Malay/Indonesian distinction is clearer

TEST: Malay example
INPUT: Saya sedang belajar bahasa Melayu.
CHECK: Malay/Indonesian distinction is clearer

TEST: Indonesian example
INPUT: Saya sedang belajar bahasa Indonesia.
CHECK: Malay/Indonesian distinction is uncertain


In [93]:
for name, sentence in ambiguity_tests.items():

    features = final_vectorizer.transform([sentence])

    prediction = final_model.predict(features)[0]

    print("\nTEST:", name)
    print("PREDICTED:", language_names[prediction])
    print("AMBIGUITY:", check_malay_indonesian_ambiguity(sentence))


TEST: Malay sentence
PREDICTED: Malay
AMBIGUITY: Malay/Indonesian distinction is clearer

TEST: Indonesian sentence
PREDICTED: Malay
AMBIGUITY: Malay/Indonesian distinction is clearer

TEST: Malay example
PREDICTED: Malay
AMBIGUITY: Malay/Indonesian distinction is clearer

TEST: Indonesian example
PREDICTED: Indonesian
AMBIGUITY: Malay/Indonesian distinction is uncertain


In [96]:
for name, sentence in ambiguity_tests.items():

    result = check_malay_indonesian_ambiguity(sentence)

    print("\nTEST:", name)
    print("STATUS:", result["status"])
    print("Malay:", result["malay_probability"], "%")
    print("Indonesian:", result["indonesian_probability"], "%")


TEST: Malay sentence
STATUS: Clearer prediction
Malay: 97.2 %
Indonesian: 2.68 %

TEST: Indonesian sentence
STATUS: Clearer prediction
Malay: 80.39 %
Indonesian: 19.12 %

TEST: Malay example
STATUS: Clearer prediction
Malay: 72.68 %
Indonesian: 26.52 %

TEST: Indonesian example
STATUS: Uncertain
Malay: 46.41 %
Indonesian: 51.06 %


In [106]:
def analyse_language_message(sentence):

    mixed_result = detect_mixed_language_v4(sentence)
    ambiguity_result = check_malay_indonesian_ambiguity(sentence)

    print("VERITAS LANGUAGELINK")
    print("-" * 35)

    print("Input:", sentence)
    print("Detection:", mixed_result["result"])
    print("Languages:", mixed_result["languages"])

    print("\nMalay probability:",
          ambiguity_result["malay_probability"], "%")

    print("Indonesian probability:",
          ambiguity_result["indonesian_probability"], "%")

    if ambiguity_result["status"] == "Uncertain":
        print("\nWarning: Malay/Indonesian distinction is uncertain.")

    elif ambiguity_result["status"] == "Not relevant":
        print("\nMalay/Indonesian ambiguity: Not applicable.")

    else:
        print("\nMalay/Indonesian prediction: Clearer.")

    if ambiguity_result["status"] == "Uncertain":
        print("\nNote: Similar languages may be difficult to distinguish.")

    elif mixed_result["result"] == "Possible mixed-language message detected":
        print("\nNote: Multiple languages detected. Results are predictions, not guaranteed.")

    else:
        print("\nNote: Only one dominant language was identified.")


print("LanguageLink analysis function ready!")

LanguageLink analysis function ready!


In [99]:
analyse_language_message(
    "I really enjoy learning because apprendre le français is interesting."
)

VERITAS LANGUAGELINK
-----------------------------------
Input: I really enjoy learning because apprendre le français is interesting.
Detection: Possible mixed-language message detected
Languages: ['English', 'French']

Malay probability: 2.02 %
Indonesian probability: 1.62 %

Additional note: Language identification may be less reliable for similar languages.


In [104]:
analyse_language_message(
    "I really enjoy learning because apprendre le français is interesting."
)

VERITAS LANGUAGELINK
-----------------------------------
Input: I really enjoy learning because apprendre le français is interesting.
Detection: Possible mixed-language message detected
Languages: ['English', 'French']

Malay probability: 2.02 %
Indonesian probability: 1.62 %

Malay/Indonesian ambiguity: Not applicable.

Note: Language identification may be less reliable for similar languages.


In [107]:
final_tests = [
    "I really enjoy learning because apprendre le français is interesting.",
    "I want to makan nasi goreng today.",
    "Saya suka bermain sepak bola bersama teman saya.",
    "I enjoy playing football with my friends every weekend."
]

for sentence in final_tests:

    print("\n" + "=" * 55)

    analyse_language_message(sentence)


VERITAS LANGUAGELINK
-----------------------------------
Input: I really enjoy learning because apprendre le français is interesting.
Detection: Possible mixed-language message detected
Languages: ['English', 'French']

Malay probability: 2.02 %
Indonesian probability: 1.62 %

Malay/Indonesian ambiguity: Not applicable.

Note: Multiple languages detected. Results are predictions, not guaranteed.

VERITAS LANGUAGELINK
-----------------------------------
Input: I want to makan nasi goreng today.
Detection: Possible mixed-language message detected
Languages: ['English', 'Indonesian']

Malay probability: 11.53 %
Indonesian probability: 14.18 %

Malay/Indonesian ambiguity: Not applicable.

Note: Multiple languages detected. Results are predictions, not guaranteed.

VERITAS LANGUAGELINK
-----------------------------------
Input: Saya suka bermain sepak bola bersama teman saya.
Detection: One dominant language detected
Languages: ['Indonesian']

Malay probability: 80.39 %
Indonesian probabil

In [108]:
import gradio as gr

def language_link_v2(sentence):

    if not sentence.strip():
        return "Please enter a message."

    # 1. Full-sentence prediction
    features = final_vectorizer.transform([sentence])
    probabilities = final_model.predict_proba(features)[0]

    best_index = probabilities.argmax()
    predicted_code = final_model.classes_[best_index]
    predicted_language = language_names[predicted_code]
    confidence = probabilities[best_index]

    # 2. Mixed-language analysis
    mixed_result = detect_mixed_language_v4(sentence)

    # 3. Malay-Indonesian ambiguity analysis
    ambiguity = check_malay_indonesian_ambiguity(sentence)

    # 4. Support routing
    if len(mixed_result["languages"]) >= 2:
        queue = "General Support (Multilingual Review)"

    elif confidence < 0.75:
        queue = "General Support"

    else:
        queue = support_queues.get(
            predicted_code, "General Support"
        )

    # 5. Build the output
    output = f"""
## VERITAS LANGUAGELINK

**Input:** {sentence}

### Language Detection
- **Full-sentence prediction:** {predicted_language}
- **Confidence:** {confidence:.2%}
- **Mixed-language analysis:** {mixed_result["result"]}
- **Detected languages:** {", ".join(mixed_result["languages"]) or "None"}

### Support Routing
**Assigned queue:** {queue}

### Reliability Information
- Malay probability: {ambiguity["malay_probability"]}%
- Indonesian probability: {ambiguity["indonesian_probability"]}%
"""

    if ambiguity["status"] == "Uncertain":
        output += "\n⚠️ Malay/Indonesian distinction is uncertain.\n"

    if (
        len(mixed_result["languages"]) == 1
        and predicted_language not in mixed_result["languages"]
    ):
        output += """
⚠️ The full-sentence prediction and chunk-based analysis disagree.
Manual review is recommended.
"""

    if len(mixed_result["languages"]) >= 2:
        output += """
⚠️ Multiple languages detected. Predictions are not guaranteed.
"""

    output += """
---
*Veritas LanguageLink is a prototype. Confidence scores are model
estimates, not guaranteed probabilities of correctness.*
"""

    return output


# Create the Gradio interface
with gr.Blocks(title="Veritas LanguageLink") as language_app:

    gr.Markdown("""
    # VERITAS LANGUAGELINK
    ### AI Language Detection & Smart Support Routing

    Enter a message to identify its language and determine
    the appropriate support queue.
    """)

    input_box = gr.Textbox(
        label="Enter your message",
        placeholder="Type your message here...",
        lines=3
    )

    with gr.Row():
        detect_button = gr.Button("Detect Language", variant="primary")
        clear_button = gr.ClearButton()

    output_box = gr.Markdown(label="Analysis Results")

    detect_button.click(
        fn=language_link_v2,
        inputs=input_box,
        outputs=output_box
    )

    input_box.submit(
        fn=language_link_v2,
        inputs=input_box,
        outputs=output_box
    )

    clear_button.add([input_box, output_box])


print("Veritas LanguageLink interface is ready!")

language_app.launch(share=True)

Veritas LanguageLink interface is ready!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4cfae3751c704d190.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
